In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.transformer_lstm_svm import Transformer_LSTM_SVM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i}/" for i in range(1)]
MODEL_NAME = "Transformer_LSTM_SVM"


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-3,
                 seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    # two-stage model
    model = Transformer_LSTM_SVM(n_classes=8, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")
        print(f"  backbone params: {n_params:,}")
        print("  --- Stage 1: training Transformer-LSTM backbone ---")

    # stage 1: train transformer-LSTM with softmax head
    with Timer(device) as stage1_timer:
        model.fit_backbone(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr,
                       batch_size=50, seed=seed, verbose=verbose)
                       # remove weight_decay arg

    if verbose:
        print("  --- Stage 2: fitting RBF-SVM on FC(8) logits ---")

    # stage 2: fit SVM on backbone's FC(8) logits (CPU-bound)
    with Timer(device=None) as stage2_timer:
        model.fit_svm(X_tr, y_tr)

    peak_mem_mb = get_gpu_peak_memory_mb(device)
    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed

    # inference timing
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # test evaluation
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(backbone)={stage1_timer.elapsed:.1f}s  "
              f"stage2(SVM)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario":  scenario_idx,
        "model":     MODEL_NAME,
        "n_train":   len(X_tr),
        "accuracy":  acc,
        "precision": p,
        "recall":    r,
        "f1":        f,
        "confusion": cm,
        "n_params":  n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

def run_scenario_multi_seed(scenario_idx, scenario_dir, device,
                            epochs=200, lr=1e-3, seeds=range(10),
                            verbose=True):
    """Run a scenario across multiple seeds, return mean ± std."""
    seed_runs = []
    for s in seeds:
        if verbose:
            print(f"\n--- Scenario {scenario_idx}, seed={s} ---")
        r = run_scenario(scenario_idx, scenario_dir, device,
                         epochs=epochs, lr=lr, seed=s,
                         verbose=False)
        seed_runs.append(r)
        if verbose:
            print(f"  seed {s}: acc={r['accuracy']:.4f}  F1={r['f1']:.4f}")

    metric_keys = ["accuracy", "precision", "recall", "f1"]
    agg = {
        "scenario":  scenario_idx,
        "model":     seed_runs[0]["model"],
        "n_train":   seed_runs[0]["n_train"],
        "n_params":  seed_runs[0]["n_params"],
        "n_seeds":   len(list(seeds)),
    }
    for k in metric_keys:
        vals = np.array([r[k] for r in seed_runs])
        agg[k] = vals.mean()
        agg[f"{k}_std"] = vals.std(ddof=1)

    agg["train_sec"] = np.mean([r["train_sec"] for r in seed_runs])
    agg["train_sec_stage1"] = np.mean([r["train_sec_stage1"] for r in seed_runs])
    agg["train_sec_stage2"] = np.mean([r["train_sec_stage2"] for r in seed_runs])
    agg["inf_ms_per_sample"] = np.mean([r["inf_ms_per_sample"] for r in seed_runs])
    agg["peak_mem_mb"] = np.mean([r["peak_mem_mb"] for r in seed_runs])
    
    # Keep individual seed results for variance analysis
    agg["per_seed_f1"] = [r["f1"] for r in seed_runs]
    agg["per_seed_acc"] = [r["accuracy"] for r in seed_runs]

    if verbose:
        print(f"\n  === Scenario {scenario_idx} aggregated over {len(list(seeds))} seeds ===")
        print(f"  F1: {agg['f1']:.4f} ± {agg['f1_std']:.4f}")
        print(f"  Acc: {agg['accuracy']:.4f} ± {agg['accuracy_std']:.4f}")

    return agg

In [3]:
SEEDS = list(range(10))   # 0 through 9

results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario_multi_seed(i, sc_dir, device,
                                epochs=200, lr=1e-3, seeds=SEEDS)
    results.append(r)


--- Scenario 1, seed=0 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 0: acc=0.9881  F1=0.9881

--- Scenario 1, seed=1 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 1: acc=0.9881  F1=0.9881

--- Scenario 1, seed=2 ---


...

  seed 7: acc=0.9900  F1=0.9900

--- Scenario 1, seed=8 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 8: acc=0.9931  F1=0.9931

--- Scenario 1, seed=9 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 9: acc=0.9900  F1=0.9900

  === Scenario 1 aggregated over 10 seeds ===
  F1: 0.9806 ± 0.0299
  Acc: 0.9806 ± 0.0297


In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "accuracy_std",
                       "precision", "precision_std",
                       "recall", "recall_std",
                       "f1", "f1_std",
                       "n_params", "train_sec",
                       "train_sec_stage1", "train_sec_stage2",
                       "inf_ms_per_sample", "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "accuracy_std",
              "precision", "precision_std",
              "recall", "recall_std",
              "f1", "f1_std"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios (mean over {len(SEEDS)} seeds) ===")
print(summary.to_string(index=False))
summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)

# Also save per-seed details
per_seed = pd.DataFrame([
    {"scenario": r["scenario"], "seed": s, "f1": f1, "acc": acc}
    for r in results
    for s, f1, acc in zip(SEEDS, r["per_seed_f1"], r["per_seed_acc"])
])
per_seed.to_csv(f"{MODEL_NAME.lower()}_per_seed.csv", index=False)


=== Transformer_LSTM_SVM summary across scenarios (mean over 10 seeds) ===
 scenario                model  n_train  accuracy  accuracy_std  precision  precision_std  recall  recall_std     f1  f1_std  n_params  train_sec  train_sec_stage1  train_sec_stage2  inf_ms_per_sample  peak_mem_mb
        1 Transformer_LSTM_SVM   546563    0.9806        0.0297     0.9812         0.0286  0.9806      0.0297 0.9806  0.0299     10869  23902.992         23568.879           334.114             1.7619      20970.9


In [5]:
# for r in results:
#     print(f"\nScenario {r['scenario']}  ({r['model']}, "
#           f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
#     print(r["confusion"])